## imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, 
                            mean_squared_error, mean_absolute_error, confusion_matrix,
                            f1_score)
from scipy import stats
import re
import ast
import warnings
from __future__ import annotations

from collections import Counter, defaultdict
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Any, Optional, Set

warnings.filterwarnings('ignore')
# Set matplotlib parameters for better looking plots
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

## create gt

In [ ]:
import pandas as pd
import numpy as np

files = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\defendants\one_defendants.csv'
df = pd.read_csv(files)

# רק יהודים וערבים
df = df[df['ethnic'].isin(['Jewish', 'Arab'])].copy()

# גודל מדגם רצוי
n_total = 125
n_per_group = n_total // 2   # 62
extra = n_total % 2          # 1, כדי להגיע ל-125

# פיצול לפי קבוצה
df_jewish = df[df['ethnic'] == 'Jewish'].copy()
df_arab = df[df['ethnic'] == 'Arab'].copy()

# אם רוצים 125 בדיוק, אפשר לתת את ה-1 הנוסף לקבוצה עם יותר תצפיות
if len(df_jewish) >= len(df_arab):
    n_jewish = n_per_group + extra   # 63
    n_arab = n_per_group             # 62
else:
    n_jewish = n_per_group
    n_arab = n_per_group + extra

def stratified_sample_by_district(data, n, district_col='district', random_state=42):
    # חלוקה פרופורציונלית לפי מחוז
    counts = data[district_col].value_counts(normalize=True)
    alloc = (counts * n).round().astype(int)

    # תיקון כדי שסך הכל יצא בדיוק n
    diff = n - alloc.sum()
    if diff != 0:
        order = (counts * n - alloc).sort_values(ascending=False)
        for idx in order.index[:abs(diff)]:
            alloc[idx] += 1 if diff > 0 else -1

    samples = []
    for district, k in alloc.items():
        district_df = data[data[district_col] == district]
        k = min(k, len(district_df))  # הגנה אם ביקשנו יותר ממה שיש
        if k > 0:
            samples.append(district_df.sample(n=k, random_state=random_state))

    result = pd.concat(samples)

    # אם בגלל min(...) יצא פחות מ-n, נשלים רנדומלית מהשאר
    if len(result) < n:
        remaining = data.drop(result.index)
        needed = n - len(result)
        if len(remaining) >= needed:
            extra_sample = remaining.sample(n=needed, random_state=random_state)
            result = pd.concat([result, extra_sample])

    # אם יצא יותר מ-n, נחתוך
    if len(result) > n:
        result = result.sample(n=n, random_state=random_state)

    return result

sample_jewish = stratified_sample_by_district(df_jewish, n_jewish, district_col='district', random_state=42)
sample_arab = stratified_sample_by_district(df_arab, n_arab, district_col='district', random_state=42)

final_sample = pd.concat([sample_jewish, sample_arab]).sample(frac=1, random_state=42).reset_index(drop=True)
# SEACRH link.split('/')[-1].split('.')[0] 
files = [link.split('/')[-1].split('.')[0] for link in final_sample['Document Link']]
#search each file in C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\all_verdicts.csv create new df like all_verdicts.csv but only with the files in the sample
all_verdicts = pd.read_csv(r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\all_verdicts.csv')
sample_verdicts = all_verdicts[all_verdicts['verdict'].isin(files)].copy()
sample_verdicts.to_csv(r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\sample_verdicts.csv', index=False)
print(len(sample_verdicts))

## create pred csv

In [ ]:
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\features_extractions.csv", encoding="utf-8-sig")
# keep only rpws when age column is under 15 make it int
#drop all the rows that have לא ידוע in age column
df = df[df['age'] != 'לא ידוע'].copy()
df['age'] = df['age'].astype(int)
df_under_15 = df[df['age'] < 15].copy()
folder_path = r'C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\CSV_OUTPUT'
texts = []
# for original_case in df_under_15['original_case'] take the text from the file in folder_path with the same name and save it in a list
for original_case in df_under_15['original_case']:
    file_path = f"{folder_path}/{original_case}.csv"
    file_df = pd.read_csv(file_path, encoding="utf-8-sig")
    text = file_df['text'].tolist()
    # combine all the text in the list to one string and save it in a new list
    text = "\n".join(text)
    texts.append(text)

# add the text to the df_under_15
df_under_15['text'] = texts
print(len(df_under_15))
columns = ['original_case', 'age', 'text']
df_under_15 = df_under_15[columns]
df_under_15.to_csv(
    r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\under_15_cases.csv",
    index=False,
    encoding="utf-8-sig"
)


## llm play

In [ ]:
FEW_SHOT_EXAMPLES = """
מטרה: לזהות את המשפט שבו השופט/ת מכריז/ת על מתחם הענישה בתיק הנוכחי, ולחלץ את גבולות המתחם בחודשי מאסר בפועל.

דוגמה 1:
טקסט: "לאחר ששקלתי את כל הנסיבות, אני קובע כי מתחם העונש ההולם נע בין 12 חודשי מאסר ל-24 חודשי מאסר."
מתחם: 12 - 24

דוגמה 2:
טקסט: "הנאשם הורשע בעבירה של החזקת נשק. הוא ירצה 6 חודשי מאסר."
מתחם: אין

דוגמה 3:
טקסט: "לטענת המדינה מתחם העונש ההולם נמצא בין שנתיים עד ארבע שנות מאסר. מכל האמור הגעתי למסקנה שמתחם העונש ההולם נמצא בין 9 חודשי מאסר בפועל עד 3 שנות מאסר בפועל."
מתחם: 9 - 36

דוגמה 4:
טקסט: "ע''פ 2482/22 מדינת ישראל נ' קדורה: נקבע מתחם הנע בין 10-36 חודשי מאסר. לאחר בחינת כלל השיקולים מצאתי כי יש לקבוע בנסיבות מקרה זה מתחם ענישה של 24-40 חודשי מאסר בפועל."
מתחם: 24 - 40

דוגמה 5 (מתחמים מרובים מאותו תיק):
טקסט: "אני קובעת כי מתחם הענישה באישום הראשון נע בין שבע לבין עשר שנות מאסר בפועל; לאישום השני מאסר בפועל בין 36 לבין 60 חודשים."
מתחם: 84 - 120

דוגמה 6 (מתחמים מתיקים שונים - לקחת רק את המתחם העיקרי):
טקסט: "לאור כל האמור אני סבורה שמתחמי הענישה שיש לקבוע במקרה זה הם כדלקמן: בתיק הנשק – מתחם כולל לשני האישומים הנע בין שתיים לארבע שנות מאסר. בתיק הסמים – מתחם כולל שבין שנה לשנתיים מאסר."
מתחם: 24 - 48

דוגמה 7 (סטטוטורי מול מתחם שנקבע בפועל):
טקסט: "מתחם העונש הינו בין 30 ל-60 חודשי מאסר... בהתאם לתיקון 113... סבורני כי מתחם העונש ההולם הינו החל מ-18 ועד 42 חודשי מאסר."
מתחם: 18 - 42

דוגמה 8 (גבול תחתון לא מספרי → ריק!):
טקסט: "לאור כל זאת, אני בדעה כי מתחם הענישה... נע בין עונש מאסר קצר שניתן לשאת בעבודות שירות ועד 12 חודשי מאסר בפועל."
מתחם: ריק - 12

דוגמה 9 (אימוץ מפורש של הצעת צד):
טקסט: "התביעה ביקשה לקבוע מתחם 8-18 חודשי מאסר. מתחם זה נמצא הולם והוא יפה לענייננו."
מתחם: 8 - 18

דוגמה 10 (חודשים + שנים בסוגריים → החודשים קובעים):
טקסט: "אני קובעת שמתחם העונש הראוי נע בין 54-78 חודשי מאסר (4.5 עד 6.5 שנים)."
מתחם: 54 - 78

דוגמה 11 (גבול תחתון לא-משמורת → 0):
טקסט: "מתחם העונש ההולם נע ממאסר על תנאי ועד 12 חודשי מאסר בפועל."
מתחם: 0 - 12
"""


def build_prompt(text: str) -> str:
    return f"""{FEW_SHOT_EXAMPLES}

כעת נתח את הטקסט הבא בזהירות רבה:

טקסט: "{text}"

מטרה:
לחלץ את מתחם הענישה הסופי (מתחם הולם) שקבע בית המשפט בתיק זה בלבד.

הוראות חשיבה (בצע את הניתוח הזה לפני מתן התשובה):
1. **סריקה:** זהה את כל הטווחים המספריים בטקסט (חודשים ושנים).
2. **סינון ראשוני (שלילה):**
   - האם המספרים מופיעים במשפט המתאר תיק אחר? (חפש מילות מפתח: "רע\"פ", "ע\"פ", "ת\"פ", "בפרשת...", "באותו עניין", "בית המשפט המחוזי קבע" בהקשר של תקדים). -> אם כן, התעלם!
   - האם המספרים הם חלק מטיעוני הצדדים? ("עתרה", "ביקשה", "לטענת", "סבורה"). -> סמן כ"מועמד להדחה" (אלא אם יאושר בשלב הבא).

3. **זיהוי הקביעה הקובעת (היררכיה):**
   - **עדיפות א' - קביעה ישירה:** חפש ביטויים בגוף ראשון כמו "אני קובע", "לדידי", "סבורני", "מצאתי לנכון", "לוז המתחם", "המתחם בתיק זה".
   - **עדיפות ב' - אימוץ (JUDGE_ADOPTION):** חפש ביטויי הסכמה מפורשים ("מקובל עלי", "יפה לענייננו", "מאמץ את עמדת...", "המתחם שהוצע... הולם").
     * אם נמצא ביטוי כזה, חפש את המספרים המוצמדים אליו או אלו שהופיעו במשפט *שמיד לפניו*.
     * בדוק שהאימוץ מתייחס למתחם הענישה ולא לעונש הספציפי.

4. **עיבוד סופי:**
   - המר שנים לחודשים.
   - אם נותרו מספר מתחמים לאישומים שונים בתיק הנוכחי -> קח את הגבול העליון הגבוה ביותר ואת הגבול התחתון הגבוה ביותר ללא סכימה של עונשים.

דגשים קריטיים למניעת טעויות:
* **כלל ה"לדידי":** המילה "לדידי" (לדעתי) היא האינדיקטור החזק ביותר לקביעת השופט. היא גוברת על הכל.
* **זהירות מציטוטים:** אם משפט מסתיים ב"באותו מקרה" – זהו תקדים חיצוני. מחק אותו מיד.
* **אימוץ:** אם השופט כותב "המתחם שהציעה המאשימה מקובל עלי" – קח את המספרים מטיעוני המאשימה.

פלט סופי (3 שורות בדיוק):
משפט רלוונטי: [הציטוט המדויק שנבחר. במקרה של אימוץ, צטט את משפט האימוץ + המשפט עם המספרים]
מתחם: [X - Y] (בחודשים בלבד. לגבי הגבול התחתון X: אם כתוב רק "של"צ"/"מאסר על תנאי" -> כתוב 0. אם מוזכר "מאסר בפועל" ללא מספר (גם אם ביחד עם של"צ/תנאי) -> כתוב "ריק". אם אין מספר כלל -> "ריק")
רמת ביטחון: [גבוהה/בינונית/נמוכה]
""".strip()

In [ ]:
import pandas as pd
df = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\CSV_OUTPUT\SH-15-03-13289-44.csv", encoding="utf-8-sig")
text = ""
for para in df['text']:
	text += para + "\n"
	
all_drugs_prompt = build_prompt(text)

In [ ]:
from openai import OpenAI
client = OpenAI()  # reads OPENAI_API_KEY from the environment

response = client.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": "אתה עוזר משפטי מומחה בניתוח פסקי דין בעברית, מתמחה בעבירות סמים."},
        {"role": "user", "content": all_drugs_prompt}
    ]
)
result = response.choices[0].message.content
result


## plot the results

In [ ]:
# ============================================================================
# CONSTANTS
# ============================================================================

MISSING = "__MISSING__"

# --- plot appearance ---
BACKGROUND_COLOR = "white"

ACADEMIC_FONT_FAMILY = "serif"
ACADEMIC_FONT = ["Times New Roman", "DejaVu Serif", "serif"]

# --- binary (yes/no) categorical columns ---
binary_cols = [
    'section_144a', 'section_144b', 'section_144b2', 'section_144b3',
    'confession', 'escape_attempt', 'has_independent_offenses',
    'premeditation', 'regret', 'undercover_sale', 'is_actual_sentence',
]

# --- multi-class categorical columns ---
multiclass_cols = [
    'Criminal_record', 'actual_damage',
    'marital_status', 'weapon_acquisition', 'weapon_motive',
    'weapon_status', 'weapon_use',
]
# 'potential_damage',
categorical_cols = binary_cols + multiclass_cols

# --- standard numeric columns (non weapon-count) ---
numeric_cols = [
    'magazines', 'bullets', 
    'age', 'punishment_months',
    'min_punishment', 'max_punishment',
]
# 'weapon_price',
# --- weapon-count columns (handled as a combined block, like old drug_cols) ---
weapon_count_cols = [
    'handgun', 'submachine_gun', 'assault_rifle', 'sniper_rifle',
    'hunting_rifle', 'improvised_firearm', 'grenade',
    'explosive_charge', 'molotov', 'anti_tank_missile',
    'ammunition_only', 'weapon_parts', 'unidentified_firearm',
]

# --- multi-label (comma-separated sets) ---
multilabel_cols = ['weapon_storage']

# sentinel strings that mean "missing/unknown" in numeric fields
NUMERIC_MISSING_TOKENS = {'לא ידוע', 'לא צוין', 'לא ברור', '', 'nan', 'NaN', 'None'}

demographic_columns = ['ethnic']


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def to_numeric_clean(series, treat_neg_one_as_missing=True):
    s_raw = series.astype(str).str.strip()
    missing_mask = s_raw.isin(NUMERIC_MISSING_TOKENS)
    out = pd.to_numeric(series, errors='coerce')
    out = out.where(~missing_mask, np.nan)
    if treat_neg_one_as_missing:
        out = out.where(out != -1, np.nan)
    return out


def parse_multilabel(value):
    if pd.isna(value):
        return set()
    s = str(value).strip().strip('"').strip("'")
    if not s or s in NUMERIC_MISSING_TOKENS:
        return set()
    return {x.strip() for x in s.split(',') if x.strip()}


def multilabel_jaccard(set_gt, set_pred):
    if not set_gt and not set_pred:
        return 1.0
    uni = len(set_gt | set_pred)
    return len(set_gt & set_pred) / uni if uni > 0 else 1.0


def multilabel_f1(set_gt, set_pred):
    if not set_gt and not set_pred:
        return 1.0
    tp = len(set_gt & set_pred)
    fp = len(set_pred - set_gt)
    fn = len(set_gt - set_pred)
    denom = 2 * tp + fp + fn
    return 2 * tp / denom if denom > 0 else 0.0


# ============================================================================
# VISUALIZATION HELPERS
# ============================================================================

def add_sig_bracket(ax, x1, x2, y, h, text, color='#0a1628', lw=1.2, fs=11):
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color=color, lw=lw, clip_on=False)
    ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom',
            color=color, fontsize=fs, fontweight='bold')


def get_stars(p_value):
    if pd.isna(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return ''


def translate_label(label):
    if label == 'Not Jewish':
        return 'Arab'
    return str(label).replace('_', ' ').title()


# ============================================================================
# METRIC FUNCTIONS
# ============================================================================

def calculate_categorical_metrics(df1, df2, col):
    y_true = df1[col].fillna(MISSING).astype(str).str.strip()
    y_pred = df2[col].fillna(MISSING).astype(str).str.strip()

    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    matches = (y_true == y_pred).astype(float).values
    n_errors = int((y_true != y_pred).sum())
    error_rate = n_errors / len(y_true) if len(y_true) > 0 else np.nan
    return {
        'accuracy': acc, 'f1': f1, 'precision': prec, 'recall': rec,
        'matches': matches, 'y_true': y_true.values, 'y_pred': y_pred.values,
        'n_errors': n_errors, 'error_rate': error_rate, 'n': len(y_true),
    }


def calculate_multilabel_metrics(df1, df2, col):
    gt_sets = df1[col].apply(parse_multilabel)
    pr_sets = df2[col].apply(parse_multilabel)

    f1s = np.array([multilabel_f1(g, p) for g, p in zip(gt_sets, pr_sets)])
    jacs = np.array([multilabel_jaccard(g, p) for g, p in zip(gt_sets, pr_sets)])
    matches = np.array([1.0 if g == p else 0.0 for g, p in zip(gt_sets, pr_sets)])

    n = len(f1s)
    n_errors = int((matches == 0).sum())
    return {
        'accuracy': float(matches.mean()) if n else np.nan,
        'f1': float(f1s.mean()) if n else np.nan,
        'precision': float(jacs.mean()) if n else np.nan,
        'recall': float(jacs.mean()) if n else np.nan,
        'matches': matches,
        'y_true': np.array([','.join(sorted(s)) for s in gt_sets]),
        'y_pred': np.array([','.join(sorted(s)) for s in pr_sets]),
        'f1_scores': f1s,
        'jaccard_scores': jacs,
        'n_errors': n_errors,
        'error_rate': n_errors / n if n else np.nan,
        'n': n,
    }


def calculate_numeric_metrics(df1, df2, col, tol=0.0, treat_neg_one_as_missing=True):
    gt = to_numeric_clean(df1[col], treat_neg_one_as_missing)
    pred = to_numeric_clean(df2[col], treat_neg_one_as_missing)
    mask = gt.notna() & pred.notna()
    if mask.sum() == 0:
        return None

    gt_v = gt[mask].astype(float)
    pr_v = pred[mask].astype(float)

    diff = pr_v - gt_v
    abs_err = diff.abs()
    sq_err = diff.pow(2)

    mae = abs_err.mean()
    rmse = np.sqrt(sq_err.mean())

    pos = gt_v > 0
    if pos.any():
        pe_values = (abs_err[pos] / gt_v[pos]) * 100
        mape = float(pe_values.mean())
    else:
        pe_values = pd.Series(dtype=float)
        mape = np.nan

    err_mask = abs_err > tol
    n_valid = int(mask.sum())
    n_err = int(err_mask.sum())
    error_rate = n_err / n_valid if n_valid else np.nan

    if n_err > 0:
        error_size_mae = abs_err[err_mask].mean()
        error_size_median = abs_err[err_mask].median()
        error_size_rmse = np.sqrt(sq_err[err_mask].mean())
        errors_only = abs_err[err_mask].values
        squared_errors_only = sq_err[err_mask].values
    else:
        error_size_mae = 0.0
        error_size_median = 0.0
        error_size_rmse = 0.0
        errors_only = np.array([])
        squared_errors_only = np.array([])

    return {
        'mae': mae, 'rmse': rmse, 'mape': mape,
        'pe_values': pe_values.values if hasattr(pe_values, 'values') else np.asarray(pe_values),
        'errors': abs_err.values, 'squared_errors': sq_err.values,
        'n': n_valid,
        'error_rate': error_rate, 'n_errors': n_err,
        'error_size_mae': error_size_mae,
        'error_size_median': error_size_median,
        'error_size_rmse': error_size_rmse,
        'errors_only': errors_only,
        'squared_errors_only': squared_errors_only,
    }


def calculate_combined_weapon_count_metrics(df1, df2, cols):
    all_gt, all_pred, all_pe = [], [], []

    for c in cols:
        gt = to_numeric_clean(df1[c], treat_neg_one_as_missing=True)
        pr = to_numeric_clean(df2[c], treat_neg_one_as_missing=True)
        valid = gt.notna() & pr.notna()
        if valid.sum() == 0:
            continue
        all_gt.extend(gt[valid].tolist())
        all_pred.extend(pr[valid].tolist())

        pos = valid & (gt > 0)
        if pos.any():
            all_pe.extend(((pr[pos] - gt[pos]).abs() / gt[pos] * 100).tolist())

    if not all_gt:
        return None

    gt_arr = np.asarray(all_gt, dtype=float)
    pr_arr = np.asarray(all_pred, dtype=float)
    diff = pr_arr - gt_arr
    abs_err = np.abs(diff)
    sq_err = diff ** 2

    mae = float(abs_err.mean())
    rmse = float(np.sqrt(sq_err.mean()))
    pe_arr = np.asarray(all_pe, dtype=float)
    mape = float(pe_arr.mean()) if pe_arr.size else np.nan

    err_mask = abs_err > 0.0
    n_valid = abs_err.size
    n_err = int(err_mask.sum())
    error_rate = n_err / n_valid if n_valid else np.nan

    if n_err > 0:
        error_size_mae = float(abs_err[err_mask].mean())
        error_size_median = float(np.median(abs_err[err_mask]))
        error_size_rmse = float(np.sqrt(sq_err[err_mask].mean()))
        errors_only = abs_err[err_mask]
        squared_errors_only = sq_err[err_mask]
    else:
        error_size_mae = 0.0
        error_size_median = 0.0
        error_size_rmse = 0.0
        errors_only = np.array([])
        squared_errors_only = np.array([])

    return {
        'mae': mae, 'rmse': rmse, 'mape': mape, 'pe_values': pe_arr,
        'errors': abs_err, 'squared_errors': sq_err,
        'n': n_valid,
        'error_rate': error_rate, 'n_errors': n_err,
        'error_size_mae': error_size_mae,
        'error_size_median': error_size_median,
        'error_size_rmse': error_size_rmse,
        'errors_only': errors_only,
        'squared_errors_only': squared_errors_only,
    }


# ============================================================================
# BOOTSTRAP CIs
# ============================================================================

def bootstrap_ci(data, alpha=0.05, n_bootstrap=3000):
    if len(data) == 0:
        return np.nan, np.nan
    boot = []
    n = len(data)
    for _ in range(n_bootstrap):
        boot.append(np.mean(np.random.choice(data, size=n, replace=True)))
    return np.percentile(boot, alpha/2*100), np.percentile(boot, (1-alpha/2)*100)


def bootstrap_ci_rmse(squared_errors, alpha=0.05, n_bootstrap=3000):
    if len(squared_errors) == 0:
        return np.nan, np.nan
    boot = []
    n = len(squared_errors)
    for _ in range(n_bootstrap):
        boot.append(np.sqrt(np.mean(np.random.choice(squared_errors, size=n, replace=True))))
    return np.percentile(boot, alpha/2*100), np.percentile(boot, (1-alpha/2)*100)


def _bootstrap_pair_metric(y_true, y_pred, metric_fn, alpha=0.05, n_bootstrap=3000):
    if len(y_true) == 0:
        return np.nan, np.nan
    boot = []
    n = len(y_true)
    idx = np.arange(n)
    for _ in range(n_bootstrap):
        s = np.random.choice(idx, size=n, replace=True)
        try:
            boot.append(metric_fn(y_true[s], y_pred[s]))
        except Exception:
            continue
    if not boot:
        return np.nan, np.nan
    return np.percentile(boot, alpha/2*100), np.percentile(boot, (1-alpha/2)*100)


def bootstrap_ci_f1(y_true, y_pred, alpha=0.05, n_bootstrap=3000):
    return _bootstrap_pair_metric(
        y_true, y_pred,
        lambda a, b: f1_score(a, b, average='macro', zero_division=0),
        alpha, n_bootstrap,
    )


def bootstrap_ci_precision(y_true, y_pred, alpha=0.05, n_bootstrap=3000):
    return _bootstrap_pair_metric(
        y_true, y_pred,
        lambda a, b: precision_recall_fscore_support(a, b, average='macro', zero_division=0)[0],
        alpha, n_bootstrap,
    )


def bootstrap_ci_recall(y_true, y_pred, alpha=0.05, n_bootstrap=3000):
    return _bootstrap_pair_metric(
        y_true, y_pred,
        lambda a, b: precision_recall_fscore_support(a, b, average='macro', zero_division=0)[1],
        alpha, n_bootstrap,
    )


# ============================================================================
# MAIN ANALYSIS
# ============================================================================

def _append_categorical_result(results, group_name, col, metrics):
    ci_lower_acc, ci_upper_acc = bootstrap_ci(metrics['matches'])
    if 'f1_scores' in metrics:
        ci_lower_f1, ci_upper_f1 = bootstrap_ci(metrics['f1_scores'])
        ci_lower_pr, ci_upper_pr = bootstrap_ci(metrics['jaccard_scores'])
        ci_lower_rc, ci_upper_rc = bootstrap_ci(metrics['jaccard_scores'])
    else:
        ci_lower_f1, ci_upper_f1 = bootstrap_ci_f1(metrics['y_true'], metrics['y_pred'])
        ci_lower_pr, ci_upper_pr = bootstrap_ci_precision(metrics['y_true'], metrics['y_pred'])
        ci_lower_rc, ci_upper_rc = bootstrap_ci_recall(metrics['y_true'], metrics['y_pred'])

    results['categorical'].append({
        'group': group_name,
        'variable': col,
        'accuracy': metrics['accuracy'],
        'accuracy_ci_lower': ci_lower_acc, 'accuracy_ci_upper': ci_upper_acc,
        'f1': metrics['f1'],
        'f1_ci_lower': ci_lower_f1, 'f1_ci_upper': ci_upper_f1,
        'precision': metrics['precision'],
        'precision_ci_lower': ci_lower_pr, 'precision_ci_upper': ci_upper_pr,
        'recall': metrics['recall'],
        'recall_ci_lower': ci_lower_rc, 'recall_ci_upper': ci_upper_rc,
        'n': metrics['n'], 'n_errors': metrics['n_errors'],
        'error_rate': metrics['error_rate'],
    })


def _append_numeric_result(results, group_name, col, metrics):
    ci_lower_mae, ci_upper_mae = bootstrap_ci(metrics['errors'])
    ci_lower_rmse, ci_upper_rmse = bootstrap_ci_rmse(metrics['squared_errors'])

    if metrics.get('pe_values') is not None and len(metrics['pe_values']) > 0:
        ci_lower_mape, ci_upper_mape = bootstrap_ci(metrics['pe_values'])
    else:
        ci_lower_mape, ci_upper_mape = np.nan, np.nan

    err_indicator = (metrics['errors'] > 0.0).astype(float)
    er_ci_l, er_ci_u = bootstrap_ci(err_indicator)

    if len(metrics['errors_only']) > 0:
        es_ci_l, es_ci_u = bootstrap_ci(metrics['errors_only'])
    else:
        es_ci_l, es_ci_u = np.nan, np.nan

    results['numeric'].append({
        'group': group_name,
        'variable': col,
        'mae': metrics['mae'],
        'mae_ci_lower': ci_lower_mae, 'mae_ci_upper': ci_upper_mae,
        'rmse': metrics['rmse'],
        'rmse_ci_lower': ci_lower_rmse, 'rmse_ci_upper': ci_upper_rmse,
        'mape': metrics.get('mape', np.nan),
        'mape_ci_lower': ci_lower_mape, 'mape_ci_upper': ci_upper_mape,
        'n': metrics['n'],
        'error_rate': metrics['error_rate'],
        'error_rate_ci_lower': er_ci_l, 'error_rate_ci_upper': er_ci_u,
        'n_errors': metrics['n_errors'],
        'error_size_mae': metrics['error_size_mae'],
        'error_size_median': metrics['error_size_median'],
        'error_size_rmse': metrics['error_size_rmse'],
        'error_size_ci_lower': es_ci_l, 'error_size_ci_upper': es_ci_u,
    })


def analyze_by_demographic(df1, df2, demo_col):
    if demo_col in df1.columns:
        groups = [g for g in df1[demo_col].unique() if not pd.isna(g)]
    else:
        groups = []

    results = {'categorical': [], 'numeric': []}
    all_groups = ['All'] + list(groups)

    for group in all_groups:
        if group == 'All':
            df1_g, df2_g, group_name = df1, df2, 'All'
        else:
            mask = df1[demo_col] == group
            df1_g = df1[mask]
            df2_g = df2[mask]
            group_name = str(group)
            if len(df1_g) < 10:
                continue

        for col in categorical_cols:
            if col not in df1_g.columns or col not in df2_g.columns:
                continue
            m = calculate_categorical_metrics(df1_g, df2_g, col)
            if m:
                _append_categorical_result(results, group_name, col, m)

        for col in multilabel_cols:
            if col not in df1_g.columns or col not in df2_g.columns:
                continue
            m = calculate_multilabel_metrics(df1_g, df2_g, col)
            if m:
                _append_categorical_result(results, group_name, col, m)

        NEG_ONE_SPECIAL_COLS = {"magazines", "bullets"}
        for col in numeric_cols:
            if col not in df1_g.columns or col not in df2_g.columns:
                continue
            m = calculate_numeric_metrics(df1_g, df2_g, col,
                                          treat_neg_one_as_missing=col not in NEG_ONE_SPECIAL_COLS)
            if m:
                _append_numeric_result(results, group_name, col, m)

        m = calculate_combined_weapon_count_metrics(df1_g, df2_g, weapon_count_cols)
        if m:
            _append_numeric_result(results, group_name, 'weapon_counts_combined', m)

    return {
        'categorical': pd.DataFrame(results['categorical']),
        'numeric': pd.DataFrame(results['numeric']),
    }


# ============================================================================
# PLOTTING
# ============================================================================

def plot_metric_comparison(df_results, metric_col, ci_lower_col, ci_upper_col,
                           demo_col, ylabel, output_file,
                           background_color=None, ylim=None):
    if len(df_results) == 0:
        print(f"Skipping {output_file} - no data")
        return

    variables = list(df_results['variable'].unique())
    groups = list(df_results['group'].unique())

    if len(variables) == 0 or len(groups) == 0:
        print(f"Skipping {output_file} - no variables or groups")
        return

    bg = background_color if background_color is not None else BACKGROUND_COLOR
    transparent_bg = bg is None or (isinstance(bg, str) and bg.lower() == "transparent")

    sns.set_theme(
        style="whitegrid",
        context="paper",
        font=ACADEMIC_FONT_FAMILY,
        rc={
            "font.family": ACADEMIC_FONT_FAMILY,
            "font.serif": ACADEMIC_FONT,
            "axes.edgecolor": "#333333",
            "axes.linewidth": 0.8,
            "axes.labelcolor": "#1a1a1a",
            "axes.titlesize": 12,
            "axes.labelsize": 12,
            "xtick.color": "#1a1a1a",
            "ytick.color": "#1a1a1a",
            "xtick.labelsize": 10,
            "ytick.labelsize": 10,
            "legend.fontsize": 10,
            "grid.color": "#dcdcdc",
            "grid.linestyle": "--",
            "grid.linewidth": 0.6,
            "axes.grid.axis": "y",
        },
    )

    base_colors = ['#7B9E87', '#B95A48', '#1A2C48', '#D4A574']
    palette = {}
    palette['All'] = base_colors[0]
    other_groups = [g for g in groups if g != 'All']
    for i, g in enumerate(other_groups):
        palette[g] = base_colors[(i + 1) % len(base_colors)]
    for g in groups:
        palette.setdefault(g, base_colors[0])

    fig, ax = plt.subplots(figsize=(16, 6))

    sns.barplot(
        data=df_results,
        x='variable', y=metric_col,
        hue='group',
        order=variables, hue_order=groups,
        palette=palette,
        edgecolor='white', linewidth=1.2,
        saturation=0.95,
        errorbar=None, ax=ax, legend=False,
    )

    n_groups = len(groups)
    width = 0.8 / n_groups

    all_values = df_results[metric_col].values
    all_ci_upper = df_results[ci_upper_col].values
    valid_ci = ~np.isnan(all_ci_upper) & ~np.isnan(all_values)
    global_max = np.nanmax(all_ci_upper[valid_ci]) if valid_ci.any() else (
        np.nanmax(all_values) if len(all_values) else 1
    )

    # scale used for bracket positioning
    bracket_scale = ylim[1] if ylim is not None else global_max

    for i, group in enumerate(groups):
        df_group = df_results[df_results['group'] == group]
        for var_idx, var in enumerate(variables):
            row = df_group[df_group['variable'] == var]
            if len(row) == 0:
                continue
            v = row[metric_col].values[0]
            cl = row[ci_lower_col].values[0]
            cu = row[ci_upper_col].values[0]
            if pd.isna(v):
                continue
            yerr_l = abs(v - cl) if not pd.isna(cl) else 0.0
            yerr_u = abs(cu - v) if not pd.isna(cu) else 0.0
            x_center = var_idx + (i - (n_groups - 1) / 2) * width
            ax.errorbar(x_center, v, yerr=[[yerr_l], [yerr_u]],
                        fmt='none', color='#2c3e50',
                        capsize=3, capthick=1.0, elinewidth=1.0)

    subgroups = [g for g in groups if g != 'All']
    if len(subgroups) == 2:
        g1_name, g2_name = subgroups[0], subgroups[1]
        g1_idx = groups.index(g1_name)
        g2_idx = groups.index(g2_name)

        for var_idx, var in enumerate(variables):
            d1 = df_results[(df_results['group'] == g1_name) & (df_results['variable'] == var)]
            d2 = df_results[(df_results['group'] == g2_name) & (df_results['variable'] == var)]
            if len(d1) == 0 or len(d2) == 0:
                continue

            m1, n1 = d1[metric_col].values[0], d1['n'].values[0]
            ci_l1, ci_u1 = d1[ci_lower_col].values[0], d1[ci_upper_col].values[0]
            m2, n2 = d2[metric_col].values[0], d2['n'].values[0]
            ci_l2, ci_u2 = d2[ci_lower_col].values[0], d2[ci_upper_col].values[0]

            if pd.isna([m1, m2, ci_l1, ci_u1, ci_l2, ci_u2]).any():
                continue

            se1 = (ci_u1 - ci_l1) / (2 * 1.96)
            se2 = (ci_u2 - ci_l2) / (2 * 1.96)
            if se1 <= 0 or se2 <= 0:
                continue

            se_diff = np.sqrt(se1**2 + se2**2)
            t_stat = (m1 - m2) / se_diff
            df_freedom = ((se1**2 + se2**2)**2) / (
                (se1**4 / max(n1 - 1, 1)) + (se2**4 / max(n2 - 1, 1))
            )
            p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df_freedom))
            stars = get_stars(p_value)

            if stars:
                x1 = var_idx + (g1_idx - (n_groups - 1) / 2) * width
                x2 = var_idx + (g2_idx - (n_groups - 1) / 2) * width
                y_max = max(ci_u1, ci_u2)
                bracket_y = y_max + bracket_scale * 0.05
                bracket_h = bracket_scale * 0.02
                add_sig_bracket(ax, x1, x2, bracket_y, bracket_h, stars,
                                color='#333333', lw=1.0, fs=10)

    if ylim is not None:
        ax.set_ylim(ylim[0], ylim[1])
        ax.set_yticks(np.arange(ylim[0], ylim[1] + 1e-9, 0.2))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1f}'))
    else:
        ax.set_ylim(bottom=0, top=global_max * 1.35)

    ax.set_xlabel("")
    ax.set_ylabel(ylabel, fontsize=12, color='#1a1a1a', labelpad=8)
    ax.set_xticks(np.arange(len(variables)))
    ax.set_xticklabels([translate_label(v) for v in variables],
                       fontsize=10, color='#1a1a1a', rotation=45, ha='right')

    sns.despine(ax=ax, top=True, right=True)

    ax.yaxis.grid(True)
    ax.xaxis.grid(False)
    ax.set_axisbelow(True)

    legend_handles = [
        plt.Rectangle((0, 0), 1, 1, facecolor=palette[g],
                      edgecolor='white', linewidth=1.0)
        for g in groups
    ]
    legend_labels = [translate_label(g) for g in groups]
    fig.legend(legend_handles, legend_labels,
               loc='upper center', bbox_to_anchor=(0.5, 0.97),
               ncol=n_groups, frameon=False, fontsize=10)

    ax.tick_params(axis='x', labelsize=10, pad=6)
    ax.tick_params(axis='y', labelsize=10)

    if transparent_bg:
        fig.patch.set_facecolor('none')
        ax.set_facecolor('none')
    else:
        fig.patch.set_facecolor(bg)
        ax.set_facecolor(bg)

    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight',
                transparent=transparent_bg,
                facecolor=fig.get_facecolor())
    plt.close()
    print(f"Saved: {output_file}")


# ============================================================================
# EXECUTION
# ============================================================================

df1 = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\GT.csv", encoding="utf-8-sig")
df2 = pd.read_csv(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\PRED.csv", encoding="utf-8-sig")

def align_on_case(df1_g, df2_g, key='original_case'):
    common = np.intersect1d(df1_g[key].dropna().unique(), df2_g[key].dropna().unique())
    df1_a = df1_g[df1_g[key].isin(common)].sort_values(key).reset_index(drop=True)
    df2_a = df2_g[df2_g[key].isin(common)].sort_values(key).reset_index(drop=True)
    return df1_a, df2_a

df1 = df1[df1['original_case'].isin(df2['original_case'])].copy()
df1, df2 = align_on_case(df1, df2, key='original_case')

for demo_col in demographic_columns:
    if demo_col not in df1.columns:
        try:
            defendants_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\defendants\one_defendants.csv"
            dem = pd.read_csv(defendants_path, encoding="utf-8-sig")
            key_col = 'original_case' if 'original_case' in dem.columns else dem.columns[0]
            dem_cols = [key_col] + [c for c in [demo_col, 'ethnic'] if c in dem.columns]
            dem = dem[dem_cols].drop_duplicates(subset=[key_col])
            if demo_col not in dem.columns and 'ethnic' in dem.columns:
                dem = dem.rename(columns={'ethnic': demo_col})
            df1 = df1.merge(dem, on=key_col, how='left')
            df2 = df2.merge(dem, on=key_col, how='left')
        except Exception as e:
            print(f"(could not attach {demo_col}: {e}) -- falling back to 'All' group only")

import os
os.makedirs('llm_results', exist_ok=True)

for demo_col in demographic_columns:
    print(f"\n=== Analyzing by {demo_col} ===")
    results = analyze_by_demographic(df1, df2, demo_col)

    results['categorical'].to_csv(f'llm_results/categorical_metrics_{demo_col}.csv',
                                  index=False, encoding='utf-8-sig')
    results['numeric'].to_csv(f'llm_results/numeric_metrics_{demo_col}.csv',
                              index=False, encoding='utf-8-sig')

    plot_metric_comparison(results['categorical'], 'accuracy',
                           'accuracy_ci_lower', 'accuracy_ci_upper',
                           demo_col, 'Accuracy', f'llm_results/accuracy_{demo_col}.png',
                           ylim=(0, 1))
    plot_metric_comparison(results['categorical'], 'f1',
                           'f1_ci_lower', 'f1_ci_upper',
                           demo_col, 'F1 Score', f'llm_results/f1_{demo_col}.png',
                           ylim=(0, 1))
    plot_metric_comparison(results['categorical'], 'precision',
                           'precision_ci_lower', 'precision_ci_upper',
                           demo_col, 'Precision', f'llm_results/precision_{demo_col}.png',
                           ylim=(0, 1))
    plot_metric_comparison(results['categorical'], 'recall',
                           'recall_ci_lower', 'recall_ci_upper',
                           demo_col, 'Recall', f'llm_results/recall_{demo_col}.png',
                           ylim=(0, 1))

    plot_metric_comparison(results['numeric'], 'mae',
                           'mae_ci_lower', 'mae_ci_upper',
                           demo_col, 'Mean Absolute Error',
                           f'llm_results/mae_{demo_col}.png')
    plot_metric_comparison(results['numeric'], 'rmse',
                           'rmse_ci_lower', 'rmse_ci_upper',
                           demo_col, 'Root Mean Squared Error',
                           f'llm_results/rmse_{demo_col}.png')
    plot_metric_comparison(results['numeric'], 'error_rate',
                           'error_rate_ci_lower', 'error_rate_ci_upper',
                           demo_col, 'Error Rate',
                           f'llm_results/error_rate_{demo_col}.png')
    plot_metric_comparison(results['numeric'], 'error_size_mae',
                           'error_size_ci_lower', 'error_size_ci_upper',
                           demo_col, 'Error Size (MAE | error)',
                           f'llm_results/error_size_{demo_col}.png')

    df_mape = results['numeric'].dropna(subset=['mape'])
    plot_metric_comparison(df_mape, 'mape',
                           'mape_ci_lower', 'mape_ci_upper',
                           demo_col, 'Mean Absolute Percentage Error (%)',
                           f'llm_results/mape_{demo_col}.png')

print("\n=== Analysis complete. All results saved. ===")


## csv for whole results

In [ ]:
import pandas as pd

# --- weapon GT label schema ---
numeric_cols = [
    'magazines', 'bullets', 'weapon_price',
    'age', 'punishment_months',
    'min_punishment', 'max_punishment',
    # weapon-count columns (one row per case per type in long format)
    'handgun', 'submachine_gun', 'assault_rifle', 'sniper_rifle',
    'hunting_rifle', 'improvised_firearm', 'grenade',
    'explosive_charge', 'molotov', 'anti_tank_missile',
    'ammunition_only', 'weapon_parts', 'unidentified_firearm',
]

categorical_cols = [
    # binary
    'section_144a', 'section_144b', 'section_144b2', 'section_144b3',
    'confession', 'escape_attempt', 'has_independent_offenses',
    'premeditation', 'regret', 'undercover_sale', 'is_actual_sentence',
    # multi-class
    'Criminal_record', 'actual_damage', 'potential_damage',
    'marital_status', 'weapon_acquisition', 'weapon_motive',
    'weapon_status', 'weapon_use',
    # multi-label (kept here so the long export captures it; compare as strings)
    'weapon_storage',
]

LABEL_COLS = numeric_cols + categorical_cols

gt_path   = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\GT.csv"
pred_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\PRED.csv"

df_gt = pd.read_csv(gt_path, encoding="utf-8-sig")
df_pr = pd.read_csv(pred_path, encoding="utf-8-sig")

TEXT_COL = "original_case"

# sanity checks: only require columns that exist in BOTH files. Missing on one
# side means we cannot compare that label - drop it rather than crash.
LABEL_COLS = [c for c in LABEL_COLS if c in df_gt.columns and c in df_pr.columns]
missing_common = [c for c in [TEXT_COL] if c not in df_gt.columns or c not in df_pr.columns]
if missing_common:
    raise ValueError(f"Missing key column in one of the files: {missing_common}")

# align by case id (robust to row order differences)
common_ids = sorted(set(df_gt[TEXT_COL]).intersection(set(df_pr[TEXT_COL])))
df_gt = df_gt[df_gt[TEXT_COL].isin(common_ids)].drop_duplicates(subset=[TEXT_COL]).sort_values(TEXT_COL).reset_index(drop=True)
df_pr = df_pr[df_pr[TEXT_COL].isin(common_ids)].drop_duplicates(subset=[TEXT_COL]).sort_values(TEXT_COL).reset_index(drop=True)

# build long format
gt_long = df_gt[[TEXT_COL] + LABEL_COLS].melt(
    id_vars=[TEXT_COL], value_vars=LABEL_COLS,
    var_name="label", value_name="gt",
)
pr_long = df_pr[[TEXT_COL] + LABEL_COLS].melt(
    id_vars=[TEXT_COL], value_vars=LABEL_COLS,
    var_name="label", value_name="pred",
)

out = gt_long.merge(pr_long, on=[TEXT_COL, "label"], how="inner")
out["match"] = out["gt"].astype(str).fillna("") == out["pred"].astype(str).fillna("")

out_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\GT_vs_PRED_long.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")

print("Saved:", out_path)
print("Rows:", len(out))


## inspect errors for a specific column

Pick a column (set `COLUMN`) and the cell prints rows where GT and PRED differ. Numeric columns are compared with `np.isclose`; multi-label columns (like `weapon_storage`) are compared as sets; everything else is compared as trimmed strings.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sqlalchemy import TEXT

# =========================
# CHOOSE COLUMN HERE
# =========================
def run_column_comparison(col, TEXT=False):
    """Compare a single column or multiple columns (if col is a list)."""
    
    # Handle both single column (string) and multiple columns (list)
    if isinstance(col, str):
        columns = [col]
    else:
        columns = col
    
    SHOW_COLS = ["original_case"]  # extra context columns to include in the output
    MAX_ROWS = None                # set an int to cap rows; None = show all

    # verdict text printout (per-error row)
    PRINT_VERDICT_TEXT = TEXT
    VERDICTS_DIR = Path(r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\all_verdicts")
    MAX_TEXT_CHARS = None          # None = full text, or set int to cap each verdict

    # =========================
    # Load & align
    # =========================
    gt_path   = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\GT.csv"
    pred_path = r"C:\Users\user\OneDrive\טל\projects\justice-tal\bias_weapon\data\PRED.csv"

    gt = pd.read_csv(gt_path, encoding="utf-8-sig")
    pr = pd.read_csv(pred_path, encoding="utf-8-sig")

    for COLUMN in columns:
        if COLUMN not in gt.columns:
            print(f"Warning: {COLUMN!r} not in GT columns, skipping")
            continue
        if COLUMN not in pr.columns:
            print(f"Warning: {COLUMN!r} not in PRED columns, skipping")
            continue

        common_ids = sorted(set(gt["original_case"]).intersection(pr["original_case"]))
        gt_aligned = (gt[gt["original_case"].isin(common_ids)]
            .drop_duplicates("original_case").sort_values("original_case").reset_index(drop=True))
        pr_aligned = (pr[pr["original_case"].isin(common_ids)]
            .drop_duplicates("original_case").sort_values("original_case").reset_index(drop=True))

        # =========================
        # Normalize per column type
        # =========================
        MULTILABEL_COLS = {"weapon_storage"}
        NUMERIC_MISSING = {"לא ידוע", "לא צוין", "לא ברור", "", "nan", "None"}

        def to_num(s):
            raw = s.astype(str).str.strip()
            out = pd.to_numeric(s, errors="coerce")
            out = out.where(~raw.isin(NUMERIC_MISSING), np.nan)
            return out.where(out != -1, np.nan)

        def to_set(v):
            if pd.isna(v):
                return frozenset()
            t = str(v).strip().strip('"').strip("'")
            if not t or t in NUMERIC_MISSING:
                return frozenset()
            return frozenset(x.strip() for x in t.split(",") if x.strip())

        gt_vals, pr_vals = gt_aligned[COLUMN], pr_aligned[COLUMN]

        is_numeric = pd.api.types.is_numeric_dtype(gt_vals) or pd.api.types.is_numeric_dtype(pr_vals)
        if COLUMN in {"magazines", "bullets", "weapon_price", "age", "punishment_months",
                    "min_punishment", "max_punishment", "range_punishment",
                    "handgun", "submachine_gun", "assault_rifle", "sniper_rifle",
                    "hunting_rifle", "improvised_firearm", "grenade", "explosive_charge",
                    "molotov", "anti_tank_missile", "ammunition_only", "weapon_parts",
                    "unidentified_firearm"}:
            is_numeric = True

        if COLUMN in MULTILABEL_COLS:
            g = gt_vals.apply(to_set)
            p = pr_vals.apply(to_set)
            diff_mask = g != p
            gt_display = g.apply(lambda s: ", ".join(sorted(s)) if s else "")
            pr_display = p.apply(lambda s: ", ".join(sorted(s)) if s else "")
        elif is_numeric:
            g = to_num(gt_vals)
            p = to_num(pr_vals)
            both_nan = g.isna() & p.isna()
            close = pd.Series(False, index=g.index)
            both_valid = g.notna() & p.notna()
            close[both_valid] = np.isclose(g[both_valid].astype(float), p[both_valid].astype(float))
            diff_mask = ~(both_nan | close)
            gt_display, pr_display = g, p
        else:
            g = gt_vals.fillna("").astype(str).str.strip()
            p = pr_vals.fillna("").astype(str).str.strip()
            diff_mask = g != p
            gt_display, pr_display = g, p

        # =========================
        # Build & print output
        # =========================
        context = [c for c in SHOW_COLS if c in gt_aligned.columns]
        out = pd.DataFrame({
            **{c: gt_aligned[c].values for c in context},
            f"PRED_{COLUMN}":   gt_display.values,
            f"GT_{COLUMN}": pr_display.values,
        })
        out = out.loc[diff_mask.values].reset_index(drop=True)

        n_total = len(gt_aligned)
        n_err = len(out)
        print(f"\n{'='*80}")
        print(f"Column: {COLUMN}")
        print(f"Compared rows: {n_total} | Errors: {n_err} | Error rate: {n_err / n_total:.2%}" if n_total else "no rows")
        print("-" * 80)

        out_print = out.head(MAX_ROWS) if MAX_ROWS is not None else out

        with pd.option_context("display.max_rows", None,
                            "display.max_colwidth", 120,
                            "display.width", 200):
            print(out_print.to_string(index=False) if n_err else "(no differences)")

        # =========================
        # Print verdict text for each diff row
        # =========================
        if PRINT_VERDICT_TEXT and n_err:
            try:
                from docx import Document
            except ImportError:
                print("\n[verdict text skipped: install python-docx -> pip install python-docx]")
            else:
                def read_verdict_text(case_id):
                    p = VERDICTS_DIR / f"{case_id}.docx"
                    if not p.exists():
                        return f"[file not found: {p}]"
                    try:
                        doc = Document(str(p))
                        return "\n".join(para.text for para in doc.paragraphs if para.text.strip())
                    except Exception as e:
                        return f"[read error: {e}]"
                if TEXT:
                    print()
                    print("=" * 80)
                    print(f"VERDICT TEXTS FOR {n_err} DIFF ROWS - {COLUMN}")
                    print("=" * 80)
                    for _, row in out_print.iterrows():
                        case = row["original_case"] if "original_case" in row else row[context[0]]
                        gt_v = row[f"GT_{COLUMN}"]
                        pr_v = row[f"PRED_{COLUMN}"]
                        text = read_verdict_text(case)
                        if MAX_TEXT_CHARS is not None and len(text) > MAX_TEXT_CHARS:
                            text = text[:MAX_TEXT_CHARS] + "\n... [truncated]"
                        print()
                        print("-" * 80)
                        print(f"CASE: {case}  |  GT: {gt_v}  |  PRED: {pr_v}")
                        print("-" * 80)
                        print(text)

In [ ]:
run_column_comparison("max_punishment")


In [ ]:
weapon_count_cols = [
    'handgun', 'submachine_gun', 'assault_rifle', 'sniper_rifle',
    'hunting_rifle', 'improvised_firearm', 'grenade',
    'explosive_charge', 'molotov', 'anti_tank_missile',
    'ammunition_only', 'weapon_parts', 'unidentified_firearm',
]
# weapon_count_cols = [
#     'weapon_parts', 'unidentified_firearm',
# ]
run_column_comparison(weapon_count_cols, TEXT=False)  # שנה ל-TEXT=True לראות גם את הטקסט